# Individual-business calendar and weather forecast experiment

## tl;dr

This notebook rebuilds the original forecasting proof of concept at **individual-business level**, matching the app's canonical metro/parent/source-category scope. It compares two calendar baselines, identity-aware Poisson regression and identity-aware HGB on forward periods. All displayed findings are calculated from the run below; no ZIP-era scores are reused.

Run notebooks **01 → 02**, migrate/import the prepared data, then launch Jupyter with the project environment:

```sh
uv sync --frozen --group notebooks
uv run --frozen --env-file .env --group notebooks jupyter lab
```

Database setup and explicit import commands: [Local development](../docs/agents/local-development.md). This notebook reads imported sources and writes a separate versioned research run. It never replaces the Streamlit production HGB bundle.

### 1. Environment

In [ ]:
import importlib.metadata
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from sitesense.business_research import (
    ExperimentConfig,
    load_models,
    load_run,
    recommend,
    run_experiment,
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 16)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})
display(
    pd.Series(
        {
            package: importlib.metadata.version(package)
            for package in ["sitesense", "numpy", "pandas", "scikit-learn"]
        },
        name="Installed version",
    )
)

## Context & Methods

The target is **recorded check-ins for one business on one calendar day**, including zero-activity days within declared imported coverage. There is no ZIP aggregation, division by cohort size, multiplication by current listing count, or allocation of an area prediction to businesses.

| Stage | Fit interval | Assessment interval |
|---|---|---|
| Select fixed model specification | 2017–2019 | 2020 validation |
| Independent forward assessment | 2018–2020 | 2021 test |
| Refit saved research estimators | 2019–2021 | Future dates only |

Each stage rebuilds its own cohort using `first_activity <= fit_start` and at least **20 training check-ins**. No held-out activity, listing status, review count, rating, or future `last_checkin` determines eligibility. Model choice is frozen on 2020 mean daily Poisson deviance **before** assessing 2021. The test remains descriptive evidence, not another selection round.

### Key assumptions

- Scope is exact canonical city/state → parent category → optional exact source category. Source tags overlap; each listing contributes its **whole** recorded activity once within the selected source cohort. Snapshot tags and coordinates label an archived cohort, not historical point-in-time membership.
- To bound runtime, select at most `MAX_BUSINESSES` eligible IDs by deterministic hash of the seed and business ID, independently for each fit. The default is 64 and the cap is 128. Show full eligible and sample counts. Results describe that sample, not a citywide market.
- Calendar shrinkage priors use the complete supported parent/city pool through the production `business_training.fit` function **before** sampling the selected source cohort.
- Weather models all fit and score the same complete accepted mapped ACIS station/day reports: minimum and maximum temperature in °C, precipitation in mm. Trace rain is zero; rejected or absent observations remain missing. No scenario fills missing historical weather.
- Calendar baselines learn all covered training days; weather candidates learn only accepted-weather training rows. The same-held-out-row comparison therefore compares candidate systems and does **not** isolate weather uplift by itself.
- Calendar baselines also score **all covered business/day rows**, including days without weather. Compare weather candidates only on the shared `accepted_weather` population and inspect the missing-weather counts alongside it.
- Every learned model includes business identity: one-hot GLM effects, categorical HGB identity, or categorical CatBoost identity. Encoders, scaling and spline knots fit training rows only. No activity lags or snapshot profile values enter features.

Check-ins are an activity proxy. Observed target-day weather permits retrospective conditional prediction; an operational future prediction needs weather known at its origin. An explicit scenario is conditional evidence, not a verified weather forecast or a causal weather effect.

The 2020/2021 periods were previously examined in earlier project work; these forward results remain exploratory.

## Data

### 2. Select the exact source cohort

Change these inputs to another imported metro/parent/source tag. `None` selects the entire parent; a partial source-tag string is not a match. Lower the sample cap only for a deliberately smaller experiment; upstream/downstream notebooks must use the same configuration.

In [ ]:
METRO_CITY, METRO_STATE = "Philadelphia", "PA"
PARENT_CATEGORY = "Restaurants"
SOURCE_CATEGORY = "Pizza"
MAX_BUSINESSES = 64
RANDOM_SEED = 42
CONFIG = ExperimentConfig(
    city=METRO_CITY,
    state=METRO_STATE,
    category=PARENT_CATEGORY,
    source_category=SOURCE_CATEGORY,
    max_businesses=MAX_BUSINESSES,
    seed=RANDOM_SEED,
)
MODELS = ("overall_mean", "month_weekday_shrunk", "poisson", "hgb")
display(pd.Series(CONFIG.__dict__, name="Exact research scope"))

### 3. Fit, validate, assess, then save

The shared engine zero-fills only declared coverage; verifies distinct business/day keys; freezes the validation choice before test assessment; and records full eligible counts, sampled IDs, accepted-weather counts and source/code/lock/payload hashes. Fixed specifications keep 2021 out of tuning. Final estimators fit the independent 2019–2021 cohort.

The output root is `data/processed/business_forecast_poc/`. A run becomes discoverable only after all payloads are written and the manifest is marked `complete`. Older ZIP artifacts are incompatible with this schema.

In [ ]:
run = run_experiment(CONFIG, MODELS, ROOT / "data/processed/business_forecast_poc")
run = load_run(run.directory)
print("Verified complete run:", run.directory.relative_to(ROOT))
display(pd.DataFrame.from_dict(run.manifest["stages"], orient="index").drop(columns="business_ids"))
display(
    pd.Series(
        {
            key: run.manifest[key]
            for key in [
                "source_signature",
                "catalog_signature",
                "code_sha256",
                "lock_sha256",
            ]
        },
        name="Verified provenance",
    )
)

## Results

### 4. Read counts, missing weather and forward metrics

`all_covered` is the full sample's calendar coverage and exists for the calendar baselines. `accepted_weather` has identical business/day keys for all candidates, so its metric differences are comparable. These two populations answer different questions and must stay labeled.

In [ ]:
display(run.metrics.sort_values(["stage", "population", "poisson_deviance"]))
selected_model = run.manifest["selected_model"]
display(
    Markdown(
        f"**Model selected on 2020 only:** `{selected_model}`. "
        "The independently rebuilt 2021 assessment did not choose this model."
    )
)

### 5. Compare daily errors on identical rows

In [ ]:
comparison = run.metrics.query("population == 'accepted_weather'").copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for stage, frame in comparison.groupby("stage", sort=False):
    axes[0].plot(frame.model, frame.poisson_deviance, marker="o", label=stage)
    axes[1].plot(frame.model, frame.mae, marker="o", label=stage)
axes[0].set(title="Same accepted-weather rows", ylabel="Mean daily Poisson deviance")
axes[1].set(title="Individual business/day counts", ylabel="MAE in recorded check-ins")
for ax in axes:
    ax.tick_params(axis="x", rotation=40)
    ax.legend()
plt.tight_layout()
plt.show()

### 6. Identify businesses and inspect uneven errors

Ranks are within this selected exact-source **sample**, using each business's total over its accepted-weather assessment days. If stations have unequal missing days, these totals represent unequal observed-day exposure; use the accompanying observations counts. A larger expected archived activity count does not predict opening success for a new site.

In [ ]:
chosen = run.manifest["selected_model"]
ranking = run.ranking.query(
    "stage == 'test' and population == 'accepted_weather' and model == @chosen"
).sort_values("predicted_rank")
profiles = pd.read_json(run.directory / "catalog.json")[["business_id", "name"]]
display(ranking.merge(profiles, on="business_id", validate="one_to_one").head(10))
display(
    run.business_metrics.query(
        "stage == 'test' and population == 'accepted_weather' and model == @chosen"
    )
    .sort_values("mae", ascending=False)
    .head(10)
)

### 7. Compare two individual businesses over calendar months

The combined chart below uses complete covered-day calendar baseline predictions and actual recorded counts for the same two archived listings. Monthly sums preserve check-in units and do not average daily values into monthly activity.

In [ ]:
business_ids = run.manifest["stages"]["test"]["business_ids"][:2]
two = run.predictions.query(
    "stage == 'test' and model == 'month_weekday_shrunk' "
    "and population == 'all_covered' and business_id in @business_ids"
).copy()
two["month"] = pd.to_datetime(two.day).dt.to_period("M").dt.to_timestamp()
monthly = two.groupby(["business_id", "month"], as_index=False)[
    ["recorded_checkins", "predicted_checkins"]
].sum()
fig, ax = plt.subplots()
for identifier, frame in monthly.groupby("business_id"):
    (line,) = ax.plot(frame.month, frame.recorded_checkins, label=f"{identifier}: recorded")
    ax.plot(
        frame.month,
        frame.predicted_checkins,
        linestyle="--",
        color=line.get_color(),
        label=f"{identifier}: projected",
    )
ax.set(title="Two businesses: calendar-month recorded and expected activity", ylabel="Check-ins")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 8. Inspect the persisted HGB specification

This is the actual final research estimator, with categorical business identity, not a ZIP model or the production app model.

In [ ]:
hgb = load_models(run.directory)["hgb"]
display(pd.Series(hgb.estimator.get_params(deep=False), name="Saved HGB parameters"))
assert len(hgb.business_ids) <= MAX_BUSINESSES < hgb.estimator.max_bins
assert hgb.estimator.categorical_features == [0, 1, 2]

### 9. Run a conditional future scenario

Predictions below use only supported existing final-fit IDs. The calendar ranking needs no weather; the selected weather model uses explicitly supplied °C/mm conditions. Passing `weather=None` to a weather model requires complete accepted observations and otherwise fails, rather than inventing weather. The example day is after the 2021 final training cutoff.

In [ ]:
FUTURE_DAY = date(2022, 7, 1)
WEATHER_SCENARIO = {"tmin_c": 20.0, "tmax_c": 30.0, "precipitation_mm": 0.0}
# Calendar models do not require weather. Learned weather models require an explicit
# scenario here, or complete accepted observed reports for a retrospective day.
display(recommend(run.directory, FUTURE_DAY, model="month_weekday_shrunk", top_n=10))
display(recommend(run.directory, FUTURE_DAY, model=chosen, weather=WEATHER_SCENARIO, top_n=10))

## Takeaways

Read the executed summary below. Use notebook [05](05_forecast_gam_catboost_comparison.ipynb) to rerun the **same experiment** with additive weather splines and CatBoost. Notebook 05 verifies this manifest and reruns its baseline models instead of splicing older metrics.

Changing source data, scope, protocol, shared engine or lockfile invalidates saved results. Rerun 03 and its dependent reports. Local joblib estimators are trusted research artifacts, not an artifact format for accepting downloaded untrusted models.

In [ ]:
chosen_metrics = run.metrics.query("model == @selected_model and population == 'accepted_weather'")[
    ["stage", "businesses", "observations", "poisson_deviance", "mae"]
]
display(chosen_metrics)
display(
    Markdown(
        f"This run selected **{selected_model}** using 2020 only. "
        f"The final sample has **{run.manifest['stages']['final']['sampled_businesses']}** "
        f"of **{run.manifest['stages']['final']['eligible_businesses']}** eligible businesses. "
        "This measures conditional daily check-in prediction within the selected source cohort; "
        "it does not establish demand at a new site, revenue, or weather causality."
    )
)